In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

from notebooks.utils.classification_results_analysis_utils import available_table, load_classification_run, load_prediction_artifact
from notebooks.utils.classification_utils import evaluate_scores
from notebooks.utils.feature_aggregation_utils import FeatureSet, load_prepared_partition, prepared_partition_path
from notebooks.utils.feature_importance_utils import importance_run_path, metric_degradation, permute_feature_by_parent, summarize_importance
from notebooks.utils.local_evidence_evaluation_utils import evaluate_local_evidence_semeval
from notebooks.utils.model_training_utils import load_model_artifact, load_model_metadata, score_feature_vector_model
from notebooks.utils.classification_experiment_utils import discover_model_artifacts, prediction_artifact_key
from notebooks.utils.streaming_classification_evaluation_utils import prediction_artifact_path

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
SOURCE_RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
IMPORTANCE_RUN_ID = "extended-permutation-importance-2026-09-25-logreg-random-forest-max600-min100"
PERMUTATION_REPEATS = 5
SEED = 42
PERMUTATION_TRAINING_WINDOW = "32"
TOP_CONFIGURATIONS_PER_OBJECTIVE = 5
MODELS_ROOT = DATA_ROOT / "experiment-results" / "classification" / "models"
# Outputs are isolated under experiment-results/classification/feature-importance/<importance_run_id>.
OUTPUT_ROOT = importance_run_path(DATA_ROOT, run_id=IMPORTANCE_RUN_ID)
SEMEVAL_DATASET = "semeval2024_task8_subtask_c_train"
SEMEVAL_ANNOTATIONS = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_train_MG.csv"


In [ ]:
results = load_classification_run(DATA_ROOT, run_id=SOURCE_RUN_ID)
evaluation_metrics = available_table(results, "evaluation_metrics")
semeval_summary = available_table(results, "semeval_summary")
semeval_boundaries = available_table(results, "semeval_boundary_summary")
for frame in (evaluation_metrics, semeval_summary, semeval_boundaries):
    if "training_window" in frame.columns:
        frame["training_window"] = frame["training_window"].astype(str)
    if "evaluation_window" in frame.columns:
        frame["evaluation_window"] = frame["evaluation_window"].astype(str)
artifact_rows = []
for artifact_path in discover_model_artifacts(MODELS_ROOT, filters={"run_id": SOURCE_RUN_ID}):
    metadata = load_model_metadata(artifact_path, expected={"run_id": SOURCE_RUN_ID})
    if str(metadata["feature_set"]) != "extended":
        continue
    artifact_rows.append({"artifact_path": str(artifact_path), "artifact_sha256": hashlib.sha256((artifact_path / "model.pkl").read_bytes()).hexdigest(), "model_family": metadata["model_family"], "scorer_llm": metadata["scorer_llm"], "feature_set": metadata["feature_set"], "training_regime": metadata["training_regime"], "training_window": str(metadata["training_window"]), "feature_columns": tuple(metadata["feature_columns"])})
artifacts = pd.DataFrame(artifact_rows)
artifact_dimensions = ["model_family", "scorer_llm", "feature_set", "training_regime", "training_window"]
binary_dimensions = [*artifact_dimensions, "evaluation_regime", "evaluation_window"]
binary_rows = evaluation_metrics.loc[(evaluation_metrics["feature_set"].eq("extended")) & (evaluation_metrics["partition"].eq("validation")) & (evaluation_metrics["dataset"].isin(["kaggle_train", "mage_train", "outfox_chatgpt_train"]))].copy()
binary_ranking = binary_rows.groupby(binary_dimensions, as_index=False).agg(mean_macro_f1=("macro_f1", "mean"), minimum_macro_f1=("macro_f1", "min"), dataset_count=("dataset", "nunique")).sort_values(["mean_macro_f1", "minimum_macro_f1"], ascending=False)
binary_candidates = binary_ranking.merge(artifacts.drop(columns=["feature_columns"]), on=artifact_dimensions, how="inner")
def binary_prediction_artifacts_available(row):
    artifact_path = Path(row.artifact_path)
    dimensions = artifact_dimensions + ["evaluation_regime", "evaluation_window"]
    selected = binary_rows.loc[(binary_rows[dimensions] == pd.Series({column: getattr(row, column) for column in dimensions})).all(axis=1)]
    return all((prediction_artifact_path(results.root, prediction_artifact_key(artifact_path, models_root=MODELS_ROOT, evaluation_regime=str(row.evaluation_regime), evaluation_window=str(row.evaluation_window), dataset=str(dataset), partition="validation")) / "data.parquet").is_file() for dataset in selected["dataset"].drop_duplicates())
binary_candidates["predictions_available"] = binary_candidates.apply(binary_prediction_artifacts_available, axis=1)
binary_candidates = binary_candidates.loc[binary_candidates["dataset_count"].eq(3) & binary_candidates["predictions_available"]].copy()
binary_candidates = binary_candidates.loc[binary_candidates["training_window"].eq(PERMUTATION_TRAINING_WINDOW)].copy()
if binary_candidates.empty:
    raise RuntimeError("No binary configurations have complete prediction coverage.")
binary_selected = binary_candidates.head(TOP_CONFIGURATIONS_PER_OBJECTIVE).assign(selection_objective="binary_macro_f1")
semeval_summary = semeval_summary.loc[semeval_summary["feature_set"].eq("extended")].copy()
semeval_boundaries = semeval_boundaries.loc[semeval_boundaries["feature_set"].eq("extended")].copy()
for frame in (semeval_summary, semeval_boundaries):
    if "model_family" not in frame.columns and "model" in frame.columns:
        frame.rename(columns={"model": "model_family"}, inplace=True)
semeval_dimensions = [*artifact_dimensions, "evaluation_dataset", "semeval_split"]
semeval_ranking = semeval_summary.drop(columns=["artifact_path"], errors="ignore").merge(semeval_boundaries[semeval_dimensions + ["mean_absolute_token_error"]], on=semeval_dimensions, how="inner").sort_values(["mg_f1", "mean_absolute_token_error"], ascending=[False, True])
semeval_ranking = semeval_ranking.loc[semeval_ranking["training_window"].eq(PERMUTATION_TRAINING_WINDOW)].copy()
semeval_selected = semeval_ranking.merge(artifacts.drop(columns=["feature_columns"]), on=artifact_dimensions, how="inner").head(TOP_CONFIGURATIONS_PER_OBJECTIVE).assign(selection_objective="semeval_mg_region_f1")
selected_configurations = pd.concat([binary_selected, semeval_selected], ignore_index=True, sort=False).drop_duplicates([*artifact_dimensions, "selection_objective"]).reset_index(drop=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
binary_ranking.to_csv(OUTPUT_ROOT / "binary_ranking_source.csv", index=False)
semeval_ranking.to_csv(OUTPUT_ROOT / "semeval_ranking_source.csv", index=False)
selected_configurations.to_csv(OUTPUT_ROOT / "selected_configurations.csv", index=False)
selected_configurations


In [ ]:
binary_observations, binary_baselines = [], []
for selection in tqdm(binary_selected.itertuples(index=False), total=len(binary_selected), desc="Binary importance", unit="configuration"):
    artifact_path = Path(selection.artifact_path)
    metadata = load_model_metadata(artifact_path, expected={"run_id": SOURCE_RUN_ID})
    fitted = load_model_artifact(artifact_path, expected={"model_family": metadata["model_family"], "scorer_llm": metadata["scorer_llm"], "feature_set": "extended", "feature_columns": metadata["feature_columns"]}, trust_artifact=True)
    configuration_id = "|".join(str(getattr(selection, column)) for column in binary_dimensions)
    for dataset in binary_rows.loc[(binary_rows[binary_dimensions] == pd.Series({column: getattr(selection, column) for column in binary_dimensions})).all(axis=1), "dataset"].drop_duplicates():
        key = prediction_artifact_key(artifact_path, models_root=MODELS_ROOT, evaluation_regime=str(selection.evaluation_regime), evaluation_window=str(selection.evaluation_window), dataset=str(dataset), partition="validation")
        predictions = load_prediction_artifact(prediction_artifact_path(results.root, key))
        baseline = evaluate_scores(predictions["label"], predictions["mg_score"], 0.5)["macro_f1"]
        binary_baselines.append({"configuration_id": configuration_id, "dataset": dataset, "metric": "macro_f1", "baseline": baseline})
        permutations = ((feature_index, feature, repeat) for feature_index, feature in enumerate(fitted.feature_columns) for repeat in range(PERMUTATION_REPEATS))
        for feature_index, feature, repeat in tqdm(permutations, total=len(fitted.feature_columns) * PERMUTATION_REPEATS, desc=f"Binary {dataset} permutations", unit="permutation", leave=False):
            seed = SEED + feature_index * PERMUTATION_REPEATS + repeat
            permuted = permute_feature_by_parent(predictions, feature=feature, seed=seed)
            scored = score_feature_vector_model(fitted, permuted)
            value = evaluate_scores(scored["label"], scored["mg_score"], 0.5)["macro_f1"]
            binary_observations.append({"configuration_id": configuration_id, "dataset": dataset, "metric": "macro_f1", "feature": feature, "repeat": repeat, "seed": seed, "baseline": baseline, "permuted": value, "degradation": metric_degradation(metric="macro_f1", baseline=baseline, permuted=value)})
binary_observations = pd.DataFrame(binary_observations)
binary_baselines = pd.DataFrame(binary_baselines)
binary_observations.to_csv(OUTPUT_ROOT / "binary_permutation_observations.csv", index=False)
binary_baselines.to_csv(OUTPUT_ROOT / "binary_baselines.csv", index=False)


In [ ]:
semeval_observations, semeval_baselines = [], []
annotations = pd.read_csv(SEMEVAL_ANNOTATIONS)
for selection in tqdm(semeval_selected.itertuples(index=False), total=len(semeval_selected), desc="SemEval importance", unit="configuration"):
    artifact_path = Path(selection.artifact_path)
    metadata = load_model_metadata(artifact_path, expected={"run_id": SOURCE_RUN_ID})
    training_window = int(metadata["training_window"])
    fitted = load_model_artifact(artifact_path, expected={"model_family": metadata["model_family"], "scorer_llm": metadata["scorer_llm"], "feature_set": "extended", "feature_columns": metadata["feature_columns"]}, trust_artifact=True)
    feature_columns = tuple(metadata["feature_columns"])
    prepared = load_prepared_partition(prepared_partition_path(DATA_ROOT, scorer_llm=str(metadata["scorer_llm"]), dataset=SEMEVAL_DATASET, split="MG", feature_set="extended", window_configuration=training_window), expected={"schema_version": 1, "scorer_llm": metadata["scorer_llm"], "dataset": SEMEVAL_DATASET, "split": "MG", "feature_set": "extended", "feature_columns": list(feature_columns), "window_configuration": str(training_window), "stride": 16}).assign(window_length=training_window)
    local_classifier = SimpleNamespace(model_name=metadata["model_family"], feature_columns=feature_columns, estimator=fitted.estimator)
    semeval_model = SimpleNamespace(llm_name=metadata["scorer_llm"], classifiers={(metadata["model_family"], training_window): local_classifier}, feature_columns=feature_columns, window_lengths=(training_window,), stride=16, top_k=20)
    configuration_id = "|".join(str(getattr(selection, column)) for column in semeval_dimensions)
    baseline_result = evaluate_local_evidence_semeval(semeval_model, DATA_ROOT / "processed", annotations, dataset=SEMEVAL_DATASET, split="MG", windows_override=prepared)
    baseline_region_f1 = baseline_result.summary["mg_f1"].iloc[0]
    baseline_boundary_mae = baseline_result.boundary_summary["mean_absolute_token_error"].iloc[0]
    for metric, baseline in (("mg_region_f1", baseline_region_f1), ("boundary_mae", baseline_boundary_mae)):
        semeval_baselines.append({"configuration_id": configuration_id, "metric": metric, "baseline": baseline})
    permutations = ((feature_index, feature, repeat) for feature_index, feature in enumerate(feature_columns) for repeat in range(PERMUTATION_REPEATS))
    for feature_index, feature, repeat in tqdm(permutations, total=len(feature_columns) * PERMUTATION_REPEATS, desc=f"SemEval {training_window} permutations", unit="permutation", leave=False):
        seed = SEED + feature_index * PERMUTATION_REPEATS + repeat
        permuted = permute_feature_by_parent(prepared, feature=feature, seed=seed)
        result = evaluate_local_evidence_semeval(semeval_model, DATA_ROOT / "processed", annotations, dataset=SEMEVAL_DATASET, split="MG", windows_override=permuted)
        values = (("mg_region_f1", baseline_region_f1, result.summary["mg_f1"].iloc[0]), ("boundary_mae", baseline_boundary_mae, result.boundary_summary["mean_absolute_token_error"].iloc[0]))
        for metric, baseline, value in values:
            semeval_observations.append({"configuration_id": configuration_id, "metric": metric, "feature": feature, "repeat": repeat, "seed": seed, "baseline": baseline, "permuted": value, "degradation": metric_degradation(metric=metric, baseline=baseline, permuted=value)})
semeval_observations = pd.DataFrame(semeval_observations)
semeval_baselines = pd.DataFrame(semeval_baselines)
semeval_observations.to_csv(OUTPUT_ROOT / "semeval_permutation_observations.csv", index=False)
semeval_baselines.to_csv(OUTPUT_ROOT / "semeval_baselines.csv", index=False)


In [ ]:
observations = pd.concat([binary_observations, semeval_observations], ignore_index=True)
importance_summary = summarize_importance(observations)
rank_stability = importance_summary.groupby(["metric", "feature"], as_index=False).agg(mean_rank=("rank", "mean"), rank_standard_deviation=("rank", "std"), configuration_count=("configuration_id", "nunique")).sort_values(["metric", "mean_rank"])
importance_summary.to_csv(OUTPUT_ROOT / "importance_summary.csv", index=False)
rank_stability.to_csv(OUTPUT_ROOT / "rank_stability.csv", index=False)
(OUTPUT_ROOT / "metadata.json").write_text(json.dumps({"importance_run_id": IMPORTANCE_RUN_ID, "source_run_id": SOURCE_RUN_ID, "selection_rule": {"top_per_objective": TOP_CONFIGURATIONS_PER_OBJECTIVE, "binary": "mean validation macro-F1", "semeval": "MG-region F1 then lower boundary MAE"}, "permutation_repeats": PERMUTATION_REPEATS, "seed": SEED, "artifacts": artifacts.drop(columns=["feature_columns"]).to_dict(orient="records"), "feature_schema": artifacts.iloc[0].feature_columns if len(artifacts) else []}, indent=2, default=str), encoding="utf-8")
for metric in ("macro_f1", "mg_region_f1", "boundary_mae"):
    plot_data = rank_stability.loc[rank_stability["metric"].eq(metric)].sort_values("mean_rank")
    figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
    sns.barplot(plot_data, x="mean_rank", y="feature", ax=axis, color="steelblue")
    axis.set_title(f"Extended feature importance rank stability: {metric}")
    axis.set_xlabel("Mean feature rank (lower is more important)")
    axis.set_ylabel("Feature")
    figure.savefig(OUTPUT_ROOT / f"{metric}_rank_stability.png", dpi=160)
    plt.show()
importance_summary
